## Clustering

In [1]:
import pandas as pd
df_all = pd.read_stata("../data/HCMST Couples Meet 2017-2022 v2.2.dta", convert_categoricals=True)
df_all_num = pd.read_stata("../data/HCMST Couples Meet 2017-2022 v2.2.dta", convert_categoricals=False)
df_cluster_num = df_all_num[['caseid_new', 'w1_section', 'w1_ppage','w1_ppgender','w1_ppeducat','w1_partyid7','w1_ppmsacat', 'w1_same_sex_couple','w1_ppincimp_cat','w1_q32','w1_max_relation_status','w1_weekly_sex_frequency', 'w1_q34', 'w1_relate_duration_in2017_years', 'w1_q19', 'w1_married']]

# df_cluster = df[['caseid_new', 'w1_section', 'w1_ppage','w1_ppgender','w1_ppeducat','w1_partyid7','w1_ppmsacat', 'w1_same_sex_couple','w1_ppincimp_cat','w1_q32','w1_year_fraction_relstart','w1_max_relation_status','w1_weekly_sex_frequency']]
df_cluster = df_all[['caseid_new', 'w1_section', 'w1_ppage','w1_ppgender','w1_ppeducat','w1_partyid7','w1_ppmsacat', 'w1_same_sex_couple','w1_ppincimp_cat','w1_q32','w1_max_relation_status','w1_weekly_sex_frequency', 'w1_q34', 'w1_relate_duration_in2017_years', 'w1_q19', 'w1_married']]


#df_cluster

/var/folders/lk/bg_vtzyj3fz74y0bzpbl63fm0000gn/T/ipykernel_52034/881802232.py:2: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df_all = pd.read_stata("../data/HCMST Couples Meet 2017-2022 v2.2.dta", convert_categoricals=True)
/var/folders/lk/bg_vtzyj3fz74y0bzpbl63fm0000gn/T/ipykernel_52034/881802232.py:3: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  df_all_num = pd.read_stata("../data/HCMST Couples Meet 2017-2022 v2.2.dta", convert_categoricals=False)


### Let's cluster on this basis. 

1/ data cleaning
2/ Data types: cat vs nums seperate : it appears clean
2/ splitting? Do we split or not ? let's ask chat. 
3/ Normalize: we need to normalyse

K prototype



## Cleaning

1/ data cleaning : What is required here? 
**We drop the records or people not in relationship.** : We no longer have nulls.

Our cols are:  
num_cols = w1_ppage, w1_weekly_sex_frequency, w1_relate_duration_in2017_years, w1_ppincimp_cat, w1_q34, w1_ppeducat
cat_cols = w1_section, w1_ppgender, w1_ppmsacat, w1_same_sex_couple, w1_q32, w1_max_relation_status, w1_q19, w1_married
ord_cols = education > num, income > num, quality of relationship > num

Most ordinal columns are placed into numerical

2 coluns needs special attention:

w1_partyid7 : map to this : 
 1,2,3 > 1 (republican strong medium and lean)
 4 > 2 (indenpendant)
 5,6,7 > 3 (dem strong medium and lean)

w1_q32, 



In [2]:
## Dropping people unpartnered, filling weekly sex freqency with median, dropping 2 rows with an empty q19

# Dropping people unpartnered,
df_cluster = df_cluster.dropna(subset=['w1_q34', 'w1_relate_duration_in2017_years'])

#filling weekly sex freqency with median, 
df_cluster['w1_weekly_sex_frequency'] = df_cluster['w1_weekly_sex_frequency'].fillna(df_cluster['w1_weekly_sex_frequency'].median())

# dropping 2 rows with an empty q19
df_cluster = df_cluster.dropna(subset=['w1_q19']) # dropping 2 rows here.

## Repeat steps for num df
df_cluster_num = df_cluster_num.dropna(subset=['w1_q34', 'w1_relate_duration_in2017_years'])
df_cluster_num['w1_weekly_sex_frequency'] = df_cluster_num['w1_weekly_sex_frequency'].fillna(df_cluster_num['w1_weekly_sex_frequency'].median())
df_cluster_num = df_cluster_num.dropna(subset=['w1_q19']) # dropping 2 rows here.


# df_cluster

## Dealing with special columns

df_cluster["w1_partyid7"].value_counts()
 1,2,3 > 1 (republican strong medium and lean)
 4 > 2 (indenpendant)
 5,6,7 > 3 (dem strong medium and lean)

==> 
1 = republican
2 = Independant
3 = democrat


#w1_q32
df_cluster["w1_q32"].value_counts()

 1, -1 -> 0
 3, 8 -> 1  
 2, 4, 5, 6 -> 2

0 = Not internet
1 = Internet Dating
2 = Internet non dating

In [3]:
df_cluster_num["w1_partyid7"].value_counts()


w1_partyid7
7.0    530
5.0    523
3.0    487
1.0    408
6.0    383
2.0    351
4.0     75
Name: count, dtype: int64

In [ ]:
## Summarizing some columns, then dropping them:

df_cluster_num["w1_partyid7"].value_counts()
df_cluster_num["w1_partyid7_grouped"] = df_cluster_num["w1_partyid7"].replace({
    1: 1, 2: 1, 3: 1,
    4: 2,
    5: 3, 6: 3, 7: 3
})
# df_cluster_num["w1_partyid7_grouped"].value_counts() # it worked

df_cluster_num["w1_q32_grouped"] = df_cluster_num["w1_q32"].replace({
    1:0, -1:0,
    3:1, 8:1,
    2:2, 4:2, 5:2, 6:2
})
#df_cluster_num["w1_q32_grouped"].value_counts() # it worked

## Dropping non summarised columns:
df_cluster_num = df_cluster_num.drop(columns=["w1_partyid7", "w1_q32"])


w1_q32
 1.0    2386
 3.0     154
 2.0      65
 8.0      54
 5.0      38
 6.0      35
 4.0      18
-1.0       7
Name: count, dtype: int64

In [9]:
df_cluster_num

,caseid_new,w1_section,w1_ppage,w1_ppgender,w1_ppeducat,w1_ppmsacat,w1_same_sex_couple,w1_ppincimp_cat,w1_max_relation_status,w1_weekly_sex_frequency,w1_q34,w1_relate_duration_in2017_years,w1_q19,w1_married,w1_partyid7_grouped,w1_q32_grouped
0,53001,1,48,2,2,0,0.0,2,2.0,1.500,1.0,3.583333,1.0,1.0,3.0,0.0
1,71609,1,68,2,3,0,0.0,2,2.0,0.250,1.0,52.750000,1.0,1.0,1.0,0.0
2,106983,1,39,1,3,1,0.0,3,2.0,1.500,1.0,17.583334,1.0,1.0,3.0,0.0
3,121759,1,54,1,2,1,0.0,3,2.0,0.625,1.0,27.416666,1.0,1.0,1.0,0.0
5,164061,1,59,1,3,1,0.0,3,2.0,0.250,1.0,23.583334,1.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3505,2967957,1,30,2,2,1,0.0,2,2.0,0.250,1.0,8.666667,1.0,1.0,3.0,0.0
3506,2968357,1,42,2,3,1,1.0,2,0.0,1.500,1.0,0.666667,2.0,0.0,1.0,2.0
3507,2968971,1,23,2,2,1,0.0,1,0.0,4.500,1.0,0.833333,2.0,0.0,3.0,0.0
3508,2969933,1,34,2,4,1,1.0,1,1.0,0.250,2.0,2.333333,1.0,0.0,3.0,0.0
